# Python Classes

- self is the current object inside an instance method, so you use it to access attributes like self.name.
- Python does not truly enforce private or public, it uses conventions like _cache for “internal” and __secret for name mangling.
- Duck typing means Python cares about whether an object can do the job, not whether it inherits from a specific class.


In [1]:
class Model:
    def __init__(self, name: str, max_tokens: int):
        self.name = name
        self.max_tokens = max_tokens

    def describe(self):
        return f"{self.name} (max {self.max_tokens} tokens)"

# Dunder Methods: The Protocol System

\_\_init\_\_: Runs when an object is created, used to initialize its attributes.

In [2]:
class Embedding:
    def __init__(self, model: str, dimension: int):
        self.model = model
        self.dimension = dimension

e = Embedding("text-embedding-3-small", 1536)
print(e)
# <__main__.Embedding object at 0x7f8b8c0a1d90>

\_\_repr\_\_ :Gives a developer-friendly string representation, mainly for debugging.

The !r in the f-string calls repr() on the value, so strings include quotes.


\_\_str\_\_: Gives a human-friendly string representation, used by print()

In [14]:
class Embedding:
    def __init__(self, model: str, dimension: int):
        self.model = model
        self.dimension = dimension

    def __repr__(self):
        return f"Embedding(model={self.model!r}, dimension={self.dimension})"

    def __str__(self):
        return f"{self.model} ({self.dimension}d)"

e = Embedding("text-embedding-3-small", 1536)
print(repr(e))  # Embedding(model='text-embedding-3-small', dimension=1536)
print(e)         # text-embedding-3-small (1536d)

Embedding(model='text-embedding-3-small', dimension=1536)
text-embedding-3-small (1536d)


\_\_eq\_\_: Defines how two objects are compared with ==.

\_\_hash\_\_: Gives the object a hash value, so it can be used in sets or as dict keys.

\_\_hash\_\_ matters because dictionaries and sets use it to find objects quickly.

In short:

- If you want an object to be a dict key or go in a set, it needs a hash value.
- If two objects are equal with ==, they must have the same hash.
- If you define __eq__ but not __hash__, Python usually makes the object unhashable to prevent bugs.

In [23]:
class ModelConfig:
    def __init__(self, name: str, temperature: float):
        self.name = name
        self.temperature = temperature

    def __eq__(self, other):
        if not isinstance(other, ModelConfig):
            return NotImplemented
        return self.name == other.name and self.temperature == other.temperature

    def __hash__(self):
        return hash((self.name, self.temperature))

config_a = ModelConfig("large-model", 0.7)
config_b = ModelConfig("large-model", 0.7)

print(config_a == config_b)  # True (without __eq__, this would be False)

# With __hash__ defined, you can use configs as dict keys
cache = {config_a: "cached result for this config"}
print(cache[config_b])  # "cached result for this config"

# And store them in sets
unique_configs = {config_a, config_b}
print(len(unique_configs))  # 1 (they are equal, so the set deduplicates)

True
cached result for this config
1


### \_\_len\_\_ and __getitem__: Making Objects Behave Like Collections
\_\_len\_\_: Lets len(obj) work.

\_\_getitem\_\_: Lets you access items with indexing, like obj[i].

When you define sequence-style __getitem__, Python can iterate by asking for index 0, then 1, and so on until IndexError. This is duck typing at work.

In [ ]:
class TextDataset:
    def __init__(self, texts: list[str], labels: list[int]):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return {"text": self.texts[index], "label": self.labels[index]}

dataset = TextDataset(
    texts=["great product", "terrible service", "okay experience"],
    labels=[1, 0, 1]
)

print(len(dataset))       # 3
print(dataset[0])         # {'text': 'great product', 'label': 1}
print(dataset[-1])        # {'text': 'okay experience', 'label': 1}

for item in dataset:
    print(item["text"])

3
{'text': 'great product', 'label': 1}
{'text': 'okay experience', 'label': 1}
great product
terrible service
okay experience


# Inheritance and Method Resolution Order

MRO is the order Python follows to find a method.

For model.predict(), Python checks:

- the object itself
- its class
- parent classes
- further ancestors

With multiple inheritance, this lookup order is called the method resolution order. You can view it with:


ClassName.__mro__
or
ClassName.mro()

In [35]:
class BaseModel:
    def predict(self, text: str):
        raise NotImplementedError

class SentimentModel(BaseModel):
    def predict(self, text: str):
        positive_words = {"good", "great", "excellent", "love"}
        words = set(text.lower().split())
        score = len(words & positive_words) / max(len(words), 1)
        return {"label": "positive" if score > 0.2 else "negative", "score": score}

model = SentimentModel()
print(model.predict("this product is great and excellent"))
# {'label': 'positive', 'score': 0.33...}

SentimentModel.mro()

{'label': 'positive', 'score': 0.3333333333333333}


[__main__.SentimentModel, __main__.BaseModel, object]

# Properties and Class/Static Methods

### @property for Computed Attributes

The @property decorator lets a method behave like an attribute. It is useful when a value is derived from other fields and should stay up to date

In [36]:
class TrainingConfig:
    def __init__(self, total_samples: int, batch_size: int, epochs: int):
        self.total_samples = total_samples
        self.batch_size = batch_size
        self.epochs = epochs

    @property
    def steps_per_epoch(self):
        return self.total_samples // self.batch_size

    @property
    def total_steps(self):
        return self.steps_per_epoch * self.epochs

config = TrainingConfig(total_samples=10000, batch_size=32, epochs=3)
print(config.steps_per_epoch)  # 312
print(config.total_steps)      # 936

config.batch_size = 64
print(config.steps_per_epoch)  # 156 (automatically recalculated)

312
936
156


### @classmethod and @staticmethod

A @classmethod receives the class itself as the first argument. The most common use is an alternative constructor

A @staticmethod is a regular function that lives in the class namespace. It does not receive self or cls. Use it when the function is closely related to the class but does not need instance or class state.

In [37]:
class Prompt:
    def __init__(self, system: str, user: str):
        self.system = system
        self.user = user

    @classmethod
    def from_template(cls, template: str, **variables):
        formatted = template.format(**variables)
        return cls(system="You are a helpful assistant.", user=formatted)

    @staticmethod
    def estimate_tokens(text: str) -> int:
        return len(text) // 4

prompt = Prompt.from_template(
    "Summarize the following article about {topic}: {text}",
    topic="RAG",
    text="Retrieval augmented generation combines..."
)

tokens = Prompt.estimate_tokens("Hello, how are you?")
print(tokens)  # 4

4


# Dataclasses

Writing __init__, __repr__, and __eq__ by hand gets repetitive when a class mostly stores data. For configuration objects, API responses, evaluation records, and metadata, Python's dataclasses module can generate that boilerplate for you.

In [39]:
from dataclasses import dataclass

@dataclass()
class ModelConfig:
    name: str
    temperature: float = 0.7
    max_tokens: int = 1024
    top_p: float = 1.0

config = ModelConfig(name="large-model")
print(config)
# ModelConfig(name='large-model', temperature=0.7, max_tokens=1024, top_p=1.0)

config2 = ModelConfig(name="large-model")
print(config == config2)  # True

ModelConfig(name='large-model', temperature=0.7, max_tokens=1024, top_p=1.0)
True


## field() for Complex Defaults

The dataclass decorator raises a ValueError for common mutable defaults like lists and dictionaries. The fix is field(default_factory=...), which creates a fresh value for each instance

In [ ]:
@dataclass
class Pipeline:
    steps: list[str] = []  # This raises an error

The default_factory parameter takes a callable such as list, dict, or a small function. It creates a fresh default for each instance. The repr=False parameter leaves a field out of the string representation, which is useful for internal caches or large data structures.

In [46]:
from dataclasses import dataclass, field

@dataclass
class Pipeline:
    name: str
    steps: list[str] = field(default_factory=list)
    metadata: dict = field(default_factory=dict)
    _cache: dict = field(default_factory=dict, repr=False)

pipeline = Pipeline(name="rag-pipeline")
pipeline.steps.append("retrieve")
pipeline.steps.append("generate")
print(pipeline)
# Pipeline(name='rag-pipeline', steps=['retrieve', 'generate'], metadata={})

Pipeline(name='rag-pipeline', steps=['retrieve', 'generate'], metadata={})


## Frozen Dataclasses for Immutable Configs

The frozen=True option makes dataclass fields read-only

In [51]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class ExperimentConfig:
    model: str
    learning_rate: float
    batch_size: int
    seed: int = 42

config = ExperimentConfig(
    model="text-classifier",
    learning_rate=2e-5,
    batch_size=16
)

# config.learning_rate = 1e-4    #FrozenInstanceError

experiments = {config: {"accuracy": 0.92}}

For eq=True and frozen=True, dataclasses generate \_\_hash\_\_ when it is safe to do so. The fields still need to be hashable at runtime. A frozen dataclass containing a list cannot be used as a dictionary key until you replace the list with a tuple or another hashable structure

In [54]:
from dataclasses import dataclass

@dataclass(frozen=True)
class BadConfig:
    model: str
    layers: list[int]

b = BadConfig("gpt", [12, 24])

print(b)             # works
print(b == b)        # works

# hash(b)            # TypeError: unhashable type: 'list'
# cache = {b: "x"}    # also fails

BadConfig(model='gpt', layers=[12, 24])
True


Fix it by using a tuple instead of a list

In [55]:
from dataclasses import dataclass

@dataclass(frozen=True)
class GoodConfig:
    model: str
    layers: tuple[int, ...]

g = GoodConfig("gpt", (12, 24))

print(hash(g))       # works
cache = {g: "x"}     # works

24780474949585836


## Post-Init Processing with \_\_post_init\_\_

Sometimes you need to validate inputs or compute derived fields after the generated \_\_init\_\_ runs

In [56]:
from dataclasses import dataclass, field

@dataclass
class TrainingRun:
    model: str
    dataset: str
    epochs: int
    batch_size: int
    total_samples: int

    steps_per_epoch: int = field(init=False)
    total_steps: int = field(init=False)

    def __post_init__(self):
        if self.batch_size <= 0:
            raise ValueError(f"batch_size must be positive, got {self.batch_size}")
        if self.epochs <= 0:
            raise ValueError(f"epochs must be positive, got {self.epochs}")

        self.steps_per_epoch = self.total_samples // self.batch_size
        self.total_steps = self.steps_per_epoch * self.epochs

run = TrainingRun(
    model="language-model",
    dataset="training-corpus",
    epochs=3,
    batch_size=32,
    total_samples=10000
)
print(run.steps_per_epoch)  # 312
print(run.total_steps)      # 936

312
936


# Abstract Base Classes: Defining Interfaces

any subclass must implement ABC methods

generate_batch has a default implementation in the base class. Subclasses inherit it, but they can override it with a more efficient version

In [58]:
from abc import ABC, abstractmethod

class BaseLLM(ABC):
    @abstractmethod
    def generate(self, prompt: str) -> str:
        ...

    @abstractmethod
    def count_tokens(self, text: str) -> int:
        ...

    def generate_batch(self, prompts: list[str]) -> list[str]:
        return [self.generate(p) for p in prompts]

# This raises TypeError: Can't instantiate abstract class
# llm = BaseLLM()

class SimpleLLM(BaseLLM):
    def __init__(self, model: str = "large-model"):
        self.model = model

    def generate(self, prompt: str) -> str:
        return f"[{self.model} response to: {prompt[:50]}...]"

    def count_tokens(self, text: str) -> int:
        return len(text) // 4  # Rough estimate

llm = SimpleLLM()
print(llm.generate("Explain RAG in one sentence"))
# [large-model response to: Explain RAG in one sentence...]

results = llm.generate_batch(["prompt 1", "prompt 2"])

[large-model response to: Explain RAG in one sentence...]


# Protocols: Structural Typing

Python also supports structural typing through Protocol. With a protocol, a class can satisfy an interface by having the right methods. It does not need to inherit from the protocol.

The @runtime_checkable decorator lets you use isinstance() with the protocol. That runtime check is shallow: it checks that required attributes exist, not that full type signatures match. Without @runtime_checkable, protocols are mainly for static type checkers such as mypy and pyright

In [62]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Predictable(Protocol):
    def predict(self, text: str) -> dict[str, object]:
        ...

class SentimentModel:
    def predict(self, text: str) -> dict[str, object]:
        positive_words = {"good", "great", "excellent"}
        words = set(text.lower().split())
        score = len(words & positive_words) / max(len(words), 1)
        return {"label": "positive" if score > 0.2 else "negative"}

class ToxicityModel:
    def predict(self, text: str) -> dict[str, object]:
        toxic_words = {"hate", "terrible", "awful"}
        words = set(text.lower().split())
        score = len(words & toxic_words) / max(len(words), 1)
        return {"label": "toxic" if score > 0.1 else "safe"}

def run_analysis(model: Predictable, text: str) -> dict[str, object]:
    return model.predict(text)

sentiment = SentimentModel()
toxicity = ToxicityModel()

print(isinstance(sentiment, Predictable))  # True (runtime check)
print(run_analysis(sentiment, "this is great"))
print(run_analysis(toxicity, "this is terrible"))

True
{'label': 'positive'}
{'label': 'toxic'}
